#1 — DataExplore MAIN TeamShared — Projet & Équipe

[Datascientest](https://datascientest.com/) project - Plant Vs Disease
---
+ Date 2022/10/

Team
----
+ Promo **Sep2022 DS Bootcamp**
+ Bertrand Chatelet (apprenant)
+ François Condominas (apprenant) <nav><a href="https://www.linkedin.com/in/francoiscondominas">LinkedIn</a> | <a href="https://github.com/fcondo56">GitHub</a></nav>
+ Jeremy Cuvelier (apprenant)
+ Zakary Saheb (mentor)
+ Romain Godet (chef de cohorte)
— Raw Data**

+ source: [kaggle - new-plant-diseases-dataset](https://www.kaggle.com/datasets/vipoooool/new-plant-diseases-dataset?resource=download)
+ gitignore data/RawData
— Program structure**

1. _Import_ **PACKAGES**
2. **MAIN** _(includes (calling) FUNCTIONS in a **sequence**)_
3. **Validated FUNCTIONS** _(all FUNCTIONS in one code cell)_

4. _others code cells for **new functions under debug** or **under approval**_



*Note:* <span style="color:green">_'Images'_</span> *environment variable set to* <span style="color:green">_.../train_</span> *folder...*

In [36]:
#2 — IMPORT PACKAGES

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline                  
print('%matplotlib inline only for notebook...')
import seaborn as sns
import pprint                                       #pprint to make some output look nicer
#pp = pprint.PrettyPrinter(indent=4)
from collections import Counter
import os
from os import path
#to manage local files         
import cv2 as cv                                    #Open CV, need to powershell install w/ the following: pip install opencv-contrib-python
from PIL import Image                               #Pillow ou PIL (Python Imaging Library)
import glob                                         #globbing
import joblib
from skimage.io import imread
from skimage.transform import resize
import plotly.express as px                         #plotly for 3D xplot
from datetime import datetime
from plant_utils import *

%matplotlib inline only for notebook...


#3 — MAIN ...for DATA EXPLORATION

In [37]:
#4 — MAIN PROGRAM SEQUENCE

start_flag()
data_path = manage_CWD()
#EXECUTE the FUNCTION JUST BELOW TO INITIATE THE DF (with basic features)
df = raw_data_to_dataframe(data_path)               #can be executed once, up to you
df_to_csv(df, 'df_plant_new2')                      #export du DataFrame en CSV

end_flag()


~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~  ~~~~~~~~~~~~~~~~~  ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
  ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~  ~~~~~~~~~~~~~~~~~~~  ~~~~~~~~~~~~~~~~~~~~~~~~~~~
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~  ~~~~~~~~~~~~~~~~~~  ~~~~~~~~~~~~~~~~~~~~~~~~~~~
      ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~  Plant Vs Disease  ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~  ~~~~~~~~~~~~~~~~~~  ~~~~~~~~~~~~~~~~~~~~~~~~~~~
  ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~  ~~~~~~~~~~~~~~~~~~~  ~~~~~~~~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~  ~~~~~~~~~~~~~~~~~  ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
... 16:42:37

---------
 * CWD * 
---------

⚠️  Variable d'environnement "Images" non définie.
   Utilisation du chemin relatif par défaut.
Current CWD: /mnt/d/business/memetics/cand/cand2026/IA/github-ref/PlantVSDisease/data/PlantVSDiseasesDataset/train
data_path: /mnt/d/business/memetics/cand/cand2026/IA/github-ref/PlantVSDisease/data/PlantVSDiseasesDataset/tr

In [44]:
#5 —  Raw data with image enhancement

from plant_transform import raw_data_with_images_to_dataframe, images_enhancement

image_path = manage_CWD_arg("Images1")
modes = { 'RGB':3, 'RGB-HE':3, 'L':1, 'L-HE':1, 'L-LHE':1, 'L-CLAHE':1}
mode="L-LHE"
lz=modes[mode]
df=raw_data_with_images_to_dataframe(image_path,mode) 
#Tests avec 1884 images. 7 sous répertoires, du repertoire 'valid', selectionnés:
#Apple___Apple_scab,Apple___Black_rot,Apple___Cedar_apple_rust,Apple___healthy
#Grape___Black_rot
#Grape___Esca_(Black_Measles),Grape___healthy

#Classes equilibrées healthy/non healthy

#Test healthy/non healthy 
#mode "L-HE": f1-score de 0.62  et 0.57 , 7 min entrainement ; accuracy=0.59
#mode="L-LHE":  f1-score   (classe 0, malade) et  (classe 1, sain), 5 min entrainement; accuracy=   
#mode="L": f1-score de 0.70  et 0.24; 6min; accuracy=0,57 
#mode="L-CLAHE": f1-score 0.66 et 0.51;  4min; accuracy=0.60 
#mode="RGB":  f1-score 0.74 et 0.58; 10 min; accur=0.68 
#mode="RGB-HE": f1-score  et  de ; 11 min; accur= 

#Accroissement de la memoire sous jupiter notebook:
#https://towardsdatascience.com/leveraging-the-power-of-jupyter-notebooks-26b4b8d7c622


---------
 * CWD * 
---------

⚠️  Variable d'environnement "Images1" non définie.
   Utilisation du chemin relatif par défaut.
Current CWD: /mnt/d/business/memetics/cand/cand2026/IA/github-ref/PlantVSDisease/data/PlantVSDiseasesDataset/train
data_path: /mnt/d/business/memetics/cand/cand2026/IA/github-ref/PlantVSDisease/data/PlantVSDiseasesDataset/train

--------------------
 * RAW DATA TO DF * 
--------------------

 --- now computing --------------------------------- Apple___Apple_scab ---
 --- now computing --------------------------------- Apple___healthy ---

------------------------
 * data/image removed * 
------------------------

Amout data/image removed from the RAW dataset: 0

NA files per Label_all count:
 Counter()
data_df["image"].shape: (3,)
len(images): 3
Enhancement:     [###] 100.0% of 3

Amout data/image read/computed with success: 3

df features: Index(['label_all', 'label_plant', 'label_disease', 'filename', 'image'], dtype='object')

--------------------
 * data_

In [45]:
#6 —  Label Encoding

from sklearn.preprocessing import LabelEncoder
#print (df.columns) 
#print (df.head())
#y=df["label_all"]
y=df["label_disease"]
y=y.apply(lambda x: "Nonehealthy" if not x.startswith('healthy') else "healthy")
#y=df["label_plant"]
print (y.unique())
#print (df["label_all"].unique())
print (y.value_counts())
X=df["image"]
encoder =  LabelEncoder()
y = encoder.fit_transform(y)
#Quelques verifs
print (y.shape,X.shape)
print (np.bincount(y))
print (sum(np.bincount(y)))
print (np.unique(y))
X_first = np.array(X)
print (y.shape,X_first.shape)
#print (X_first[0].shape)
#print (X_first[0])
#X=np.reshape(X_first,(X_fisrt.shape[0],256,256,3))
print (X_first.shape)
#X = np.zeros((X_first.shape[0],256,256,3), dtype='uint8')
#print (X.shape)
#for i in range(X_first.shape[0]):
#    X[i,:,:,:]=X_first[i] 
X=np.array(X_first.tolist(),dtype='uint8')
print (X.shape)
#print(X[0,0:1,0:1,:]) 
#if lz==3 : 
#    for i in range(X.shape[0]):
#        X[i,:,:,0:4]=X[i,:,:,0:4]/255
#print(X[17571,0:1,0:1,:])

['Nonehealthy' 'healthy']
label_disease
healthy        2
Nonehealthy    1
Name: count, dtype: int64
(3,) (3,)
[1 2]
3
[0 1]
(3,) (3,)
(3,)
(3, 256, 256)


In [46]:
#7 —  Train/Test split + Undersampling

from sklearn.model_selection import train_test_split
from keras.utils import to_categorical

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20,random_state=66)

# Reshape images en 2D pour RandomUnderSampler, puis reshape inverse
n_train = X_train.shape[0]
X_flat = X_train.reshape(n_train, -1)

from imblearn.under_sampling import RandomUnderSampler
rUs = RandomUnderSampler()
X_flat_ru, y_ru = rUs.fit_resample(X_flat, y_train)

# Reshape inverse vers la forme d'origine
X_ru = X_flat_ru.reshape((-1, *X_train.shape[1:]))

y_train = to_categorical(y_ru)
y_test = to_categorical(y_test)
X_train = X_ru

In [47]:
#8 —  Model Architecture

from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Dropout, Flatten, Dense
from tensorflow.keras.models import Model

if lz==1: 
    inputs = Input(shape = (256, 256,1), name = "Input")
elif lz==3:
    inputs = Input(shape = (256, 256,3), name = "Input")

first_layer = Conv2D(filters = 32,
                     kernel_size = (3, 3),
                     padding = 'valid',
                     activation = 'relu')

second_layer = MaxPooling2D(pool_size = (2, 2))
third_layer = Dropout(rate = 0.2)
fourth_layer = Conv2D(filters = 32,
                     kernel_size = (3, 3),
                     padding = 'valid',
                     activation = 'relu')
fifth_layer = MaxPooling2D(pool_size = (2, 2))
sixth_layer=Dropout(rate = 0.2)
seventh_layer = Flatten()
output_layer = Dense(units = len(np.unique(y)),
                     activation='softmax')
#reseau à 2 couches pour discriminer 2 classes
x=first_layer(inputs)
x=second_layer(x)
x=third_layer(x)
x=fourth_layer(x)
x=fifth_layer(x)
x=sixth_layer(x)
x=seventh_layer(x)
outputs=output_layer(x)

model = Model(inputs = inputs, outputs = outputs)

In [48]:
#9 —  Model Training

from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from sklearn.utils.class_weight import compute_class_weight
early_stopping = EarlyStopping(
                                patience=5, # Attendre 5 epochs avant application
                                min_delta = 0.01, # si au bout de 5 epochs la fonction de perte ne varie pas de 1%, 
    # que ce soit à la hausse ou à la baisse, on arrête
                                verbose=1, # Afficher à quel epoch on s'arrête
                                mode = 'min',
                                monitor='val_loss')
mycallbacks = [early_stopping ]
model.compile(loss='categorical_crossentropy', # fonction de perte
              optimizer='adam',                # algorithme d'optimisation
              metrics=['accuracy'])            # métrique d'évaluation
y_train=pd.DataFrame(y_train)
#print (y.shape) 
#class_weights = compute_class_weight('balanced',y=y,classes=np.unique(y))
#class_weights = dict(enumerate(class_weights))
#L'argument class_weight, en argument de model.fit, ne marche pas (f1-scores incoherents)
#print (class_weights) 
training_history = model.fit(X_train, y_train,
                             validation_split = 0.2,
                             epochs = 25,
                             batch_size = 200,
                             callbacks=[mycallbacks])
                             #,
                             #class_weight=class_weights)

Epoch 1/25
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 1.0000 - loss: 0.6703 - val_accuracy: 0.0000e+00 - val_loss: 0.8874
Epoch 2/25
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 166ms/step - accuracy: 1.0000 - loss: 0.1912 - val_accuracy: 0.0000e+00 - val_loss: 1.6252
Epoch 3/25
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 176ms/step - accuracy: 1.0000 - loss: 0.0318 - val_accuracy: 0.0000e+00 - val_loss: 3.1408
Epoch 4/25
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 190ms/step - accuracy: 1.0000 - loss: 0.0031 - val_accuracy: 0.0000e+00 - val_loss: 5.1784
Epoch 5/25
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 152ms/step - accuracy: 1.0000 - loss: 2.2087e-04 - val_accuracy: 0.0000e+00 - val_loss: 7.3947
Epoch 6/25
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 146ms/step - accuracy: 1.0000 - loss: 1.8239e-05 - val_accuracy: 0.0000e+00 - val_loss: 9.6560
Epoch 6: early stopping


In [49]:
#10 —  Model Evaluation

from sklearn import metrics

train_acc = training_history.history['accuracy']
val_acc = training_history.history['val_accuracy']
test_pred = model.predict(X_test)

test_pred_class = test_pred.argmax(axis = 1)
y_test_class = y_test.argmax(axis = 1)

print(metrics.classification_report(y_test_class, test_pred_class, zero_division=0))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       0.0
           1       0.00      0.00      0.00       1.0

    accuracy                           0.00       1.0
   macro avg       0.00      0.00      0.00       1.0
weighted avg       0.00      0.00      0.00       1.0

